In [ ]:
import networkx as nx
import random
from scipy.stats import ks_2samp  # For Kolmogorov-Smirnov D-statistic
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
import concurrent.futures
from scipy.sparse.linalg import svds
!pip install networkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 105.3 MB/s eta 0:00:00


In [ ]:
#data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SNACS/Project/ph-dates.txt", sep=" ", header=None, names=["citing", "cited", "date"])
# Limit import to the first two columns
data = pd.read_csv('/content/dataset 2_ CA-AstroPh.txt',
                   sep="\t",
                   header=None,
                   names=["citing", "cited"],
                   comment='#')  # Skip comment lines starting with #

# Drop any duplicates (though this file should have unique edges by design)
data['citing'] = data['citing'].astype(str)
data['cited'] = data['cited'].astype(str)

data = data.drop_duplicates()

# Create a directed graph and add edges
G = nx.DiGraph()
for _, row in data.iterrows():
    if row['citing'] != row['cited']:  # Skip self-loops
        G.add_edge(row['citing'], row['cited'])

# Check the final counts
node_count = G.number_of_nodes()
edge_count = G.number_of_edges()

print(f"Expected Nodes: 34546, Actual Nodes: {node_count}")
print(f"Expected Edges: 421578, Actual Edges: {edge_count}")

Expected Nodes: 34546, Actual Nodes: 18771
Expected Edges: 421578, Actual Edges: 396100


In [ ]:
import random
import networkx as nx
from concurrent.futures import ProcessPoolExecutor


def forest_fire_sampling(graph, sample_size):
    burn_fraction = min(0.1 + (sample_size / len(graph.nodes())) * 0.1, 0.2)

    initial_burn_probability = min(0.1 + (sample_size / len(graph.nodes())) * 0.2, 0.3)

    # Fine-tune burn_probability_decay
    burn_probability_decay = max(0.02, 0.08 * (sample_size / len(graph.nodes())))  # Slower decay rate

    num_seeds = max(10, int(0.05 * sample_size))
    sampled_nodes = set()
    initial_seeds = random.sample(list(graph.nodes()), num_seeds)
    sampled_nodes.update(initial_seeds)
    to_burn = list(initial_seeds)
    current_burn_probability = initial_burn_probability

    while len(sampled_nodes) < sample_size and to_burn:
        next_to_burn = []
        for burning_node in to_burn:
            neighbors = list(graph.neighbors(burning_node))
            num_to_burn = int(len(neighbors) * burn_fraction)
            sampled_neighbors = random.sample(neighbors, min(num_to_burn, len(neighbors)))
            for neighbor in sampled_neighbors:
                if neighbor not in sampled_nodes and random.random() < current_burn_probability:
                    sampled_nodes.add(neighbor)
                    next_to_burn.append(neighbor)

        to_burn = next_to_burn
        current_burn_probability = max(current_burn_probability - burn_probability_decay, 0.05)

    return graph.subgraph(sampled_nodes).copy()


def random_walk_sampling(graph, sample_size):
    sampled_nodes = set()
    num_starting_points = 5
    max_steps = 100
    restart_probability = 0.1  # Slightly lower to explore more deeply

    all_nodes = list(graph.nodes())  # Precompute node list

    for _ in range(num_starting_points):
        node = random.choice(all_nodes)
        walk_steps = 0
        while len(sampled_nodes) < sample_size and walk_steps < max_steps:
            sampled_nodes.add(node)
            neighbors = list(graph.neighbors(node))
            if not neighbors or random.random() < restart_probability:
                node = random.choice(all_nodes)  # Restart
            else:
                node = random.choice(neighbors)  # Walk to neighbor
            walk_steps += 1

    # Return the subgraph of sampled nodes
    return graph.subgraph(sampled_nodes).copy()



def random_node_sampling(G, sample_size):
    nodes = random.sample(list(G.nodes()), sample_size)  # Convert nodes to a list
    return nx.DiGraph(G.subgraph(nodes))


def pagerank_node_sampling(G, sample_size):
    pagerank = nx.pagerank(G)
    nodes = random.choices(list(pagerank.keys()), weights=list(pagerank.values()), k=sample_size)
    return nx.DiGraph(G.subgraph(nodes))  # Ensure directed subgraph

def random_edge_sampling(G, sample_size):
    edges = random.sample(list(G.edges()), sample_size)  # Convert edges to a list
    sample_graph = nx.DiGraph()
    sample_graph.add_edges_from(edges)
    return sample_graph


def degree_distributions(G):
    in_degrees = normalize_distribution([d for _, d in G.in_degree()])
    out_degrees = normalize_distribution([d for _, d in G.out_degree()])
    return log_normalize(in_degrees), log_normalize(out_degrees)
def degree_distributions1(G):
    in_deg = np.array([d for n, d in G.in_degree()])
    out_deg = np.array([d for n, d in G.out_degree()])

    # Avoid log(0) by adding a small constant
    min_degree = min(np.min(in_deg), np.min(out_deg)) + 1  # Add 1 to avoid log(0)
    max_degree = max(np.max(in_deg), np.max(out_deg))

    bins = np.logspace(np.log10(min_degree), np.log10(max_degree), num=30)

    in_hist, _ = np.histogram(in_deg, bins=bins, density=True)
    out_hist, _ = np.histogram(out_deg, bins=bins, density=True)

    return in_hist, out_hist

def log_min_max_normalize(data):
    # Apply log transformation to compress range
    # Adding a small constant (e.g., 1e-6) to avoid log(0)
    log_data = np.log1p(np.abs(data) + 1e-6) * np.sign(data)

    # Min-max normalize the log-transformed data
    min_val = np.min(log_data)
    max_val = np.max(log_data)
    normalized_data = (log_data - min_val) / (max_val - min_val) if max_val != min_val else log_data
    return normalized_data

def log_normalize(data):
    data = np.log1p(data)  # Use log(1 + x) to handle zero values
    return min_max_normalize(data)

def min_max_normalize(data):
    min_val = min(data)
    max_val = max(data)
    return [(val - min_val) / (max_val - min_val) for val in data] if max_val != min_val else data

from scipy.stats import rankdata

def rank_normalize(data):
    return rankdata(data) / len(data)

def rank_normalize1(vector):
    ranks = np.argsort(np.argsort(vector))  # Get rank-based transformation
    normalized_vector = ranks / (len(vector) - 1)  # Scale to [0, 1] range
    return normalized_vector


def ks_test(original, sample):
    return ks_2samp(original, sample)[0]

def z_score_normalize(data):
    mean = np.mean(data)
    std = np.std(data)
    return (data - mean) / std if std > 0 else data

def normalize_distribution(values, base=np.e):
  return values
def normalize_distribution1(values, base=np.e):
    # Apply log transformation, handling zeros by adding a small constant (1) before log
    log_values = [np.log10(value + 1) / np.log10(base) for value in values]  # Adding 1 to avoid log(0)
    # Normalize the log-transformed values
    total = sum(log_values)
    if total > 0:
        return [value / total for value in log_values]
    else:
        # If total is zero, return zeros
        return [0] * len(log_values)

def calculate_hop_plot(graph, sampled_nodes, max_hops=15):
    """Calculate hop plot up to max_hops for sampled nodes within the original graph."""
    hop_counts = [0] * (max_hops + 1)  # Initialize a list to store reachable pairs count up to max_hops

    for node in sampled_nodes:
        lengths = nx.single_source_shortest_path_length(graph, node, cutoff=max_hops)
        for h in range(1, max_hops + 1):
            hop_counts[h] += sum(1 for dist in lengths.values() if dist == h)

    # Convert counts to cumulative counts (hop plot)
    hop_plot = [sum(hop_counts[:h + 1]) for h in range(max_hops + 1)]
    return hop_plot[1:]  # Skip hop 0 as it represents the node itself



def approximate_diameter_bfs(graph, sample_size=10, num_runs=10):
    total_diameter = 0

    for _ in range(num_runs):
        nodes = list(graph.nodes)
        sample_nodes = random.sample(nodes, sample_size)
        max_distance = 0

        for node in sample_nodes:
            # Run BFS for each sampled node and get the max path length
            path_lengths = nx.single_source_shortest_path_length(graph, node)
            max_distance = max(max_distance, max(path_lengths.values()))

        # Add this run's diameter to the total
        total_diameter += max_distance

    # Calculate and return the average diameter
    average_diameter = total_diameter / num_runs
    return average_diameter



def calculate_hop_counts_for_node(node):
    path_lengths = nx.single_source_shortest_path_length(G1, node, cutoff=hop_cutoff)  # Use cutoff if needed
    node_hop_counts = defaultdict(int)
    for hop_distance in path_lengths.values():
        node_hop_counts[hop_distance] += 1
    return node_hop_counts

# Parallel execution for each node
def calculate_hop_plot_parallel(G):
    max_workers=4
    hop_counts = defaultdict(int)

    # Prepare the function with partial to pass only the node to executor.map
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
          results = executor.map(calculate_hop_counts_for_node, G.nodes)


    # Aggregate results
    for node_hop_counts in results:
        for hop, count in node_hop_counts.items():
            hop_counts[hop] += count

    # Calculate cumulative reachability for the hop-plot
    max_hop = max(hop_counts.keys())
    cumulative_reachable_pairs = []
    cumulative_count = 0

    for h in range(max_hop + 1):
        cumulative_count += hop_counts[h]
        cumulative_reachable_pairs.append(cumulative_count)

    return min_max_normalize(cumulative_reachable_pairs), hop_cutoff

def first_left_singular_vector(graph):
    # Convert the graph to an adjacency matrix
    adjacency_matrix = nx.to_numpy_array(graph)

    # Perform SVD on the full adjacency matrix
    u, s, vt = svds(adjacency_matrix, k=1, which='LM')
    first_singular_vector = u[:, 0]  # Get the first left singular vector
    first_singular_vector = np.sort(first_singular_vector)[::-1]

    # Normalize the singular vector using min-max normalization
    normalized_vector = rank_normalize(first_singular_vector)
    return normalized_vector  # Return the first left singular vector

def calculate_singular_values(graph, sample_percentage, max_singular_values=30):
    # Determine the number of singular values based on sample size
    num_nodes = len(graph)
    sample_size = int(sample_percentage * num_nodes)
    num_singular_values = max(10, int(sample_percentage * max_singular_values))  # Scale with sample

    # Calculate singular values
    u, s, vt = svds(nx.to_numpy_array(graph), k=num_singular_values, which='LM')
    # Step 3: Sort the singular values in descending order
    singular_values = np.sort(s)[::-1]
    return rank_normalize(singular_values)

def clustering_coefficient_distribution1(graph):
    degree_to_clustering = {}
    for node in graph.nodes():
        k = graph.degree(node)
        if k > 1:  # Only calculate for nodes with degree > 1
            Cv = nx.clustering(graph, node)  # Local clustering coefficient for node
            if k not in degree_to_clustering:
                degree_to_clustering[k] = []
            degree_to_clustering[k].append(Cv)
        else:
            # Assign a default clustering coefficient of zero for nodes with degree < 2
            if k not in degree_to_clustering:
                degree_to_clustering[k] = []
            degree_to_clustering[k].append(0)

    # Calculate average clustering coefficient for each degree
    Cd = {degree: sum(clustering_list) / len(clustering_list)
          for degree, clustering_list in degree_to_clustering.items() if clustering_list}
    #return z_score_normalize(Cd)
    return min_max_normalize(Cd.values())

from scipy.stats import zscore

def z_score_normalize_dict(clustering_distribution):
    values = np.array(list(clustering_distribution.values()))  # Extract values from the dictionary
    normalized_values = zscore(values)  # Apply z-score normalization
    return normalized_values

def clustering_coefficient_distribution(graph):
    # Compute clustering coefficients
    clustering_coeffs = nx.clustering(graph)

    # Group by degree
    degree_to_coeffs = {}
    for node, coeff in clustering_coeffs.items():
        degree = graph.degree[node]
        if degree > 1:  # Only consider nodes with more than one neighbor
            if degree not in degree_to_coeffs:
                degree_to_coeffs[degree] = []
            degree_to_coeffs[degree].append(coeff)

    # Calculate the average clustering coefficient for each degree
    clustering_distribution = {degree: np.mean(coeffs) for degree, coeffs in degree_to_coeffs.items()}
    return z_score_normalize_dict(clustering_distribution)

import networkit as nk
import networkx as nx
from networkit import nxadapter
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor

# Define a function to calculate the effective diameter for a single graph
def calculate_effective_diameter_for_year(args):
    graph, year = args  # Unpack arguments
    ratio = 0.9  # Set ratio to 90% for effective diameter calculation

    # Ensure the graph is undirected and get the largest connected component
    undirected_graph = graph.to_undirected()
    largest_cc_nodes = max(nx.connected_components(undirected_graph), key=len)
    largest_cc_subgraph = undirected_graph.subgraph(largest_cc_nodes).copy()

    # Convert to NetworKit graph format
    nk_graph = nxadapter.nx2nk(largest_cc_subgraph)

    # Initialize and run the EffectiveDiameter algorithm
    ed = nk.distance.EffectiveDiameter(nk_graph, ratio=ratio)
    ed.run()

    # Retrieve the effective diameter
    eff_diameter = ed.getEffectiveDiameter()
    print(f"Effective diameter for {year}: {eff_diameter}")
    return year, eff_diameter
def effective_diameter_distribution(graph, percentile=90):
# Prepare the graphs by year as a list of tuples for parallel processing
    graph_items = [(graph, year) for year, graph in graph.items()]

    # Process each year’s graph in parallel and gather results
    effective_diameters = {}
    with ProcessPoolExecutor() as executor:
        results = executor.map(calculate_effective_diameter_for_year, graph_items)
        for year, eff_diameter in results:
            effective_diameters[year] = eff_diameter
    return effective_diameters

def centrality_importance_sampling(graph,sample_size,centrality_name):
    if centrality_name == 'degree':
        centrality = nx.degree_centrality(graph)
    elif centrality_name == 'betweenness':
        centrality = nx.betweenness_centrality(graph)
    elif centrality_name == 'closeness':
        centrality = nx.closeness_centrality(graph)
    else:
        raise ValueError(f"This centrality type is not defined in our program or does not exist: {centrality_name}")

    summation_of_centrality = sum(centrality.values())
    probabilities = {node: score / summation_of_centrality for node, score in centrality.items()}

    sampled_nodes = np.random.choice(list(graph.nodes), size=sample_size, replace=False, p=list(probabilities.values()))
    return graph.subgraph(sampled_nodes)

def get_weakly_connected_component_sizes(graph, sampled_nodes):
    """Calculate sizes of weakly connected components for the sampled nodes in the original graph."""
    # Extract the subgraph induced by sampled nodes, only checking their connectivity
    subgraph = graph.subgraph(sampled_nodes).to_undirected()
    component_sizes = [len(c) for c in nx.connected_components(subgraph)]
    return min_max_normalize(component_sizes)

import numpy as np
from scipy.stats import entropy

def compute_kl_divergence1(P, Q):
    """
    Computes the KL divergence between two distributions P and Q over their overlapping range.

    Parameters:
        P (list or numpy array): First distribution (counts or probabilities).
        Q (list or numpy array): Second distribution (counts or probabilities).

    Returns:
        float: KL divergence D_KL(P || Q).
    """
    # Convert to numpy arrays
    P = np.array(P, dtype=np.float64)
    Q = np.array(Q, dtype=np.float64)

    # Truncate to the overlapping range
    min_len = min(len(P), len(Q))
    P = P[:min_len]
    Q = Q[:min_len]

    # Normalize to convert counts to probabilities
    P = P / np.sum(P)
    Q = Q / np.sum(Q)

    # Avoid division by zero by replacing zeros in Q with a small constant
    Q = np.where(Q == 0, 1e-10, Q)

    # Compute KL divergence
    kl_divergence = entropy(P, Q)
    return kl_divergence

import numpy as np
from scipy.stats import entropy

def compute_kl_divergence(dist1, dist2, num_bins=30, epsilon=1e-10):
    """
    Computes the KL divergence between two distributions after preprocessing them.

    Parameters:
        dist1 (list or numpy array): First distribution (e.g., clustering coefficients, in-degree).
        dist2 (list or numpy array): Second distribution.
        num_bins (int): Number of bins to use for histogramming the distributions.
        epsilon (float): Small constant to replace zeros in the second distribution.

    Returns:
        float: KL divergence D_KL(dist1 || dist2).
    """
    dist1 = np.array(dist1, dtype=np.float64)
    dist2 = np.array(dist2, dtype=np.float64)
    # Step 1: Ensure non-negativity by shifting the distributions
    dist1_shifted = dist1 - np.min(dist1)
    dist2_shifted = dist2 - np.min(dist2)

    # Step 2: Bin the data into histograms
    bins = np.linspace(0, max(np.max(dist1_shifted), np.max(dist2_shifted)), num=num_bins)
    P, _ = np.histogram(dist1_shifted, bins=bins, density=True)
    Q, _ = np.histogram(dist2_shifted, bins=bins, density=True)

    # Step 3: Handle zeros in Q
    Q = np.where(Q == 0, epsilon, Q)

    # Step 4: Compute KL divergence
    kl_divergence = entropy(P, Q)

    return kl_divergence


In [ ]:
G1= G
hop_cutoff= int(round(approximate_diameter_bfs(G)/5))
num_sample_sizes = 1
original_in_deg, original_out_deg = degree_distributions(G)
original_wcc_sizes = min_max_normalize([len(c) for c in nx.weakly_connected_components(G)])
original_scc_sizes = min_max_normalize([len(c) for c in nx.strongly_connected_components(G)])
largest_wcc = G.subgraph(max(nx.weakly_connected_components(G), key=len)).copy()
print(largest_wcc.number_of_nodes())
original_hop_plot = calculate_hop_plot_parallel(largest_wcc)[0]
original_first_left_vector = first_left_singular_vector(G)  # For full graph
original_singular_vector = calculate_singular_values(G, sample_percentage=1)  # For full graph
Original_ce = clustering_coefficient_distribution(G)


17903


In [ ]:
!pip install pyemd


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.6/662.6 kB 8.9 MB/s eta 0:00:00


The bench mark

In [ ]:
from sklearn.cluster import SpectralClustering
import random
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor



in_deg_results = []
out_deg_results = []
wcc_results = []
scc_results = []
hops_results = []
sng_val_results = []
sng_vec_results = []
clustering_results = []
kl_in_results = []
kl_out_results = []
kl_wcc_results = []
kl_scc_results = []
kl_hops_results = []
kl_sng_val_results = []
kl_sng_vec_results = []
kl_clustering_results = []

def centrality_importance_sampling(graph,sample_size, init_sample_percentage=None,centrality_name='degree',random_seed=None):

    if random_seed is not None:
        np.random.seed(random_seed)

    if init_sample_percentage is None:
        init_sample_percentage = 80

    init_sample_percentage = int(init_sample_percentage)

    init_sample_size = int(len(graph) * (init_sample_percentage / 100))
    initial_sample = np.random.choice(list(graph.nodes()), size=init_sample_size,replace=False)
    sampled_subgraph = graph.subgraph(initial_sample)

    if centrality_name == 'degree':
        centrality = nx.degree_centrality(sampled_subgraph)
    elif centrality_name == 'betweenness':
        centrality = nx.betweenness_centrality(sampled_subgraph)
    elif centrality_name == 'closeness':
        centrality = nx.closeness_centrality(sampled_subgraph)
    else:
        raise ValueError(f"Unknown centrality: {centrality_name}")

    summation_of_centrality = sum(centrality.values())
    probabilities = np.array([score / summation_of_centrality for score in centrality.values()])
    sampled_nodes = np.random.choice(list(sampled_subgraph.nodes), size=sample_size, p=probabilities)
    return graph.subgraph(sampled_nodes)

def execute_sampling_iteration(G, sample_size, sample_percentage, centrality_name='betweenness', reuse_subgraph=None, init_sample_percentage=None):
    if reuse_subgraph:
        centrality_importance_sample = reuse_subgraph
    else:
        seed = random.randint(0, 2**32 - 1)
        centrality_importance_sample = centrality_importance_sampling(G, sample_size, init_sample_percentage=init_sample_percentage, centrality_name=centrality_name, random_seed=seed)

    in_deg_hist, out_deg_hist = degree_distributions(centrality_importance_sample)
    ks_in = ks_test(original_in_deg, in_deg_hist)
    kl_in = compute_kl_divergence(original_in_deg, in_deg_hist)  # KL Divergence for in-degree
    ks_out = ks_test(original_out_deg, out_deg_hist)
    kl_out = compute_kl_divergence(original_out_deg, out_deg_hist)  # KL Divergence for out-degree

    wcc_sample = min_max_normalize([len(c) for c in nx.weakly_connected_components(centrality_importance_sample)])
    ks_wcc = ks_test(original_wcc_sizes, wcc_sample)
    kl_wcc = compute_kl_divergence(original_wcc_sizes, wcc_sample)  # KL Divergence for WCC

    scc_sample = min_max_normalize([len(c) for c in nx.strongly_connected_components(centrality_importance_sample)])
    ks_scc = ks_test(original_scc_sizes, scc_sample)
    kl_scc = compute_kl_divergence(original_scc_sizes, scc_sample)  # KL Divergence for SCC

    # Handle global variables for hop plot
    global G1, hop_cutoff
    G1 = centrality_importance_sample.subgraph(max(nx.weakly_connected_components(centrality_importance_sample), key=len)).copy()
    hop_cutoff = int(round(approximate_diameter_bfs(G1) / 5))
    hops_sample = calculate_hop_plot_parallel(G1)[0]
    ks_hops = ks_test(original_hop_plot, hops_sample)
    kl_hops = compute_kl_divergence(original_hop_plot, hops_sample)  # KL Divergence for hops

    first_left_vector = first_left_singular_vector(centrality_importance_sample)
    ks_sng_val = ks_test(original_first_left_vector, first_left_vector)
    kl_sng_val = compute_kl_divergence(original_first_left_vector, first_left_vector)  # KL Divergence for singular value

    singular_vector = calculate_singular_values(centrality_importance_sample, sample_percentage=sample_percentage / 100)
    ks_sng_vec = ks_test(original_singular_vector, singular_vector)
    kl_sng_vec = compute_kl_divergence(original_singular_vector, singular_vector)  # KL Divergence for singular vector

    clustering_distribution = clustering_coefficient_distribution(centrality_importance_sample)
    ks_clustering = ks_test(Original_ce, clustering_distribution)
    kl_clustering = compute_kl_divergence(Original_ce, clustering_distribution)  # KL Divergence for clustering

    print("------------------------------")
    print(f"Run results for sample size {sample_size} and {sample_percentage}%:")
    print(f"In-Degree KS: {ks_in}, KL: {kl_in}")
    print(f"Out-Degree KS: {ks_out}, KL: {kl_out}")
    print(f"WCC KS: {ks_wcc}, KL: {kl_wcc}")
    print(f"SCC KS: {ks_scc}, KL: {kl_scc}")
    print(f"Hops KS: {ks_hops}, KL: {kl_hops}")
    print(f"Sng-Value KS: {ks_sng_val}, KL: {kl_sng_val}")
    print(f"Sng-Vec KS: {ks_sng_vec}, KL: {kl_sng_vec}")
    print(f"Clustering KS: {ks_clustering}, KL: {kl_clustering}")
    print("------------------------------")

    result = {
        'ks_in': ks_in,
        'ks_out': ks_out,
        'ks_wcc': ks_wcc,
        'ks_scc': ks_scc,
        'ks_hops': ks_hops,
        'ks_sng_val': ks_sng_val,
        'ks_sng_vec': ks_sng_vec,
        'ks_clustering': ks_clustering,
        'kl_in': kl_in,
        'kl_out': kl_out,
        'kl_wcc': kl_wcc,
        'kl_scc': kl_scc,
        'kl_hops': kl_hops,
        'kl_sng_val': kl_sng_val,
        'kl_sng_vec': kl_sng_vec,
        'kl_clustering': kl_clustering
    }

    in_deg_results.append(ks_in)
    out_deg_results.append(ks_out)
    wcc_results.append(ks_wcc)
    scc_results.append(ks_scc)
    hops_results.append(ks_hops)
    sng_val_results.append(ks_sng_val)
    sng_vec_results.append(ks_sng_vec)
    clustering_results.append(ks_clustering)
    kl_in_results.append(kl_in)
    kl_out_results.append(kl_out)
    kl_wcc_results.append(kl_wcc)
    kl_scc_results.append(kl_scc)
    kl_hops_results.append(kl_hops)
    kl_sng_val_results.append(kl_sng_val)
    kl_sng_vec_results.append(kl_sng_vec)
    kl_clustering_results.append(kl_clustering)


    return result

# executor = ProcessPoolExecutor(max_workers=None)
# futures = []

# initial_sample_percentage = 80
# final_sample_percentage = 100
# num_repeats = 1
# step_percentage = 5

# centralities = ['degree', 'closeness', 'betweenness']

# for centrality_name in centralities:
#     print(f"Running sampling for {centrality_name} centrality")

#     for sample_percentage in range(initial_sample_percentage, final_sample_percentage + 1, step_percentage):

#       sample_size = int((sample_percentage / 100) * len(G.nodes()))

#       print(f"Running sampling for {sample_percentage}% of the original graph (Sample size: {sample_size})")

#       for _ in range(num_repeats):
#         future = executor.submit(execute_sampling_iteration, G, sample_size, sample_percentage,centrality_name='betweenness',init_sample_percentage=sample_percentage)
#         futures.append(future)

# concurrent.futures.wait(futures)

# for future in futures:
#     (ks_in, ks_out, ks_wcc, ks_scc, ks_hops, ks_sng_val, ks_sng_vec, ks_clustering,
#      kl_in, kl_out, kl_wcc, kl_scc, kl_hops, kl_sng_val, kl_sng_vec, kl_clustering) = future.result()

#     in_deg_results.append(ks_in)
#     out_deg_results.append(ks_out)
#     wcc_results.append(ks_wcc)
#     scc_results.append(ks_scc)
#     hops_results.append(ks_hops)
#     sng_val_results.append(ks_sng_val)
#     sng_vec_results.append(ks_sng_vec)
#     clustering_results.append(ks_clustering)

#     kl_in_results.append(kl_in)
#     kl_out_results.append(kl_out)
#     kl_wcc_results.append(kl_wcc)
#     kl_scc_results.append(kl_scc)
#     kl_hops_results.append(kl_hops)
#     kl_sng_val_results.append(kl_sng_val)
#     kl_sng_vec_results.append(kl_sng_vec)
#     kl_clustering_results.append(kl_clustering)

# final_average_results_ws = {
#     'In-Degree': np.mean(in_deg_results),
#     'Out-Degree': np.mean(out_deg_results),
#     'WCC': np.mean(wcc_results),
#     'SCC': np.mean(scc_results),
#     'HOPS': np.mean(hops_results),
#     'sng-val': np.mean(sng_val_results),
#     'sng-vec': np.mean(sng_vec_results),
#     'clustering': np.nanmean(clustering_results)
# }

# print("Final Average KS Statistics:")
# print(final_average_results_ws)


# final_average_results_kl = {
#     'In-Degree': np.mean(kl_in_results),
#     'Out-Degree': np.mean(kl_out_results),
#     'WCC': np.mean(kl_wcc_results),
#     'SCC': np.mean(kl_scc_results),
#     'HOPS': np.mean(kl_hops_results),
#     'sng-val': np.mean(kl_sng_val_results),
#     'sng-vec': np.mean(kl_sng_vec_results),
#     'clustering': np.nanmean(kl_clustering_results)
# }

# print("Final Average KL Statistics:")
# print(final_average_results_kl)



In [ ]:
#original
initial_sample_percentage = 60
num_sample_sizes= 9
num_repeats = 1  # Number of repetitions for each sample size

# Initialize an accumulator for the overall KS statistics for each sampling method
overall_ks_accumulator = {'RN': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'RPN': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'RE': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'RW': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'FF': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'DCI': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'BCI': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                         'CCI': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]}
                          }

overall_kl_accumulator = {'RN': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'RPN': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'RE': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'RW': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'FF': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'DCI': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'BCI': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]},
                          'CCI': {'In-Degree': [], 'Out-Degree': [], 'WCC':[], 'SCC':[], 'HOPS':[], 'sng-val':[], 'sng-vec':[], 'clustering':[]}
                          }




for i in range(num_sample_sizes):
    # Calculate the sample size for this iteration
    sample_percentage = initial_sample_percentage + i * 5  # 5%, 10%, 15%, ..., 100%
    sample_size = int((sample_percentage / 100) * len(G.nodes()))

    print(f"Running sampling for {sample_percentage}% of the original graph (Sample size: {sample_size})")

    # Repeat the sampling and evaluation 10 times for this sample size
    for _ in range(num_repeats):
        # Perform sampling
        rn_sample = random_node_sampling(G, sample_size)
        rpn_sample = pagerank_node_sampling(G, sample_size)
        re_sample = random_edge_sampling(G, sample_size)
        rw_sample = random_walk_sampling(G, sample_size)
        ff_sample = forest_fire_sampling(G, sample_size)
        dci_sample = centrality_importance_sampling(G, sample_size, init_sample_percentage=80, centrality_name='degree')
        bci_sample = centrality_importance_sampling(G, sample_size, init_sample_percentage=80, centrality_name='betweenness')
        cci_sample = centrality_importance_sampling(G, sample_size, init_sample_percentage=80, centrality_name='closeness')

        # Calculate original degree distributions (if not already calculated)

        # For each sample, calculate distributions and compare using D-statistic
        samples = {'RN': rn_sample, 'RPN': rpn_sample, 'RE': re_sample, 'RW':rw_sample, 'FF': ff_sample, 'DCI': dci_sample, 'BCI':bci_sample, 'CCI':cci_sample}

        for name, sample in samples.items():
            in_deg_hist, out_deg_hist = degree_distributions(sample)
            ks_in = ks_test(original_in_deg, in_deg_hist)
            kl_in = compute_kl_divergence(original_in_deg, in_deg_hist)
            ks_out = ks_test(original_out_deg, out_deg_hist)
            kl_out = compute_kl_divergence(original_out_deg, out_deg_hist)

            wcc_sample = min_max_normalize([len(c) for c in nx.weakly_connected_components(sample)])
            ks_wcc = ks_test(original_wcc_sizes, wcc_sample)
            kl_wcc = compute_kl_divergence(original_wcc_sizes, wcc_sample)

            scc_sample= min_max_normalize([len(c) for c in nx.strongly_connected_components(sample)])
            ks_scc = ks_test(original_scc_sizes, scc_sample)
            kl_scc = compute_kl_divergence(original_scc_sizes, scc_sample)

            G1 = sample.subgraph(max(nx.weakly_connected_components(sample), key=len)).copy()
            hop_cutoff= int(round(approximate_diameter_bfs(G)/5))
            hops_sample = calculate_hop_plot_parallel(G1)[0]
            ks_hops = ks_test(original_hop_plot, hops_sample)
            kl_hops = compute_kl_divergence(original_hop_plot, hops_sample)

            first_left_vector = first_left_singular_vector(sample)
            ks_sng_val = ks_test(original_first_left_vector, first_left_vector)
            kl_sng_val = compute_kl_divergence(original_first_left_vector, first_left_vector)

            singular_vector = calculate_singular_values(sample, sample_percentage=sample_percentage/100)
            ks_sng_vec = ks_test(original_singular_vector, singular_vector)
            kl_sng_vec = compute_kl_divergence(original_singular_vector, singular_vector)

            clustering_distribution = clustering_coefficient_distribution(sample)
            ks_clustering = ks_test(Original_ce, clustering_distribution)
            kl_clustering = compute_kl_divergence(Original_ce, clustering_distribution)

            print("------------------------------")
            print(f"Run results for Sampling method {name} sample size {sample_size} and {sample_percentage}%:")
            print(f"In-Degree KS: {ks_in}")
            print(f"Out-Degree KS: {ks_out}")
            print(f"WCC KS: {ks_wcc}")
            print(f"SCC KS: {ks_scc}")
            print(f"Hops KS: {ks_hops}")
            print(f"Sng-Value KS: {ks_sng_val}")
            print(f"Sng-Vec KS: {ks_sng_vec}")
            print(f"Clustering KS: {ks_clustering}")
            print("------------------------------")


            print("------------------------------")
            print(f"Run results for Sampling method {name} sample size {sample_size} and {sample_percentage}%:")
            print(f"In-Degree Kl: {kl_in}")
            print(f"Out-Degree Kl: {kl_out}")
            print(f"WCC Kl: {kl_wcc}")
            print(f"SCC Kl: {kl_scc}")
            print(f"Hops Kl: {kl_hops}")
            print(f"Sng-Value Kl: {kl_sng_val}")
            print(f"Sng-Vec Kl: {kl_sng_vec}")
            print(f"Clustering Kl: {kl_clustering}")
            print("------------------------------")

            # Accumulate the KS statistics across all sizes and runs
            overall_ks_accumulator[name]['In-Degree'].append(ks_in)
            overall_ks_accumulator[name]['Out-Degree'].append(ks_out)
            overall_ks_accumulator[name]['WCC'].append(ks_wcc)
            overall_ks_accumulator[name]['SCC'].append(ks_scc)
            overall_ks_accumulator[name]['HOPS'].append(ks_hops)
            overall_ks_accumulator[name]['sng-val'].append(ks_sng_val)
            overall_ks_accumulator[name]['sng-vec'].append(ks_sng_vec)
            overall_ks_accumulator[name]['clustering'].append(ks_clustering)


            overall_kl_accumulator[name]['In-Degree'].append(kl_in)
            overall_kl_accumulator[name]['Out-Degree'].append(kl_out)
            overall_kl_accumulator[name]['WCC'].append(kl_wcc)
            overall_kl_accumulator[name]['SCC'].append(kl_scc)
            overall_kl_accumulator[name]['HOPS'].append(kl_hops)
            overall_kl_accumulator[name]['sng-val'].append(kl_sng_val)
            overall_kl_accumulator[name]['sng-vec'].append(kl_sng_vec)
            overall_kl_accumulator[name]['clustering'].append(kl_clustering)

# Calculate the final average KS statistics over all sample sizes and runs
final_average_results = {}
for name in overall_ks_accumulator:
    final_average_results[name] = {
        'In-Degree': sum(overall_ks_accumulator[name]['In-Degree']) / len(overall_ks_accumulator[name]['In-Degree']),
        'Out-Degree': sum(overall_ks_accumulator[name]['Out-Degree']) / len(overall_ks_accumulator[name]['Out-Degree']),
        'WCC': sum(overall_ks_accumulator[name]['WCC']) / len(overall_ks_accumulator[name]['WCC']),
        'SCC': sum(overall_ks_accumulator[name]['SCC']) / len(overall_ks_accumulator[name]['SCC']),
        'HOPS': sum(overall_ks_accumulator[name]['HOPS']) / len(overall_ks_accumulator[name]['HOPS']),
        'sng-val': sum(overall_ks_accumulator[name]['sng-val']) / len(overall_ks_accumulator[name]['sng-val']),
        'sng-vec': sum(overall_ks_accumulator[name]['sng-vec']) / len(overall_ks_accumulator[name]['sng-vec']),
        'clustering': np.nanmean(overall_ks_accumulator[name]['clustering'])

    }

final_average_results_kl = {}
for name in overall_kl_accumulator:
    final_average_results_kl[name] = {
        'In-Degree': sum(overall_kl_accumulator[name]['In-Degree']) / len(overall_kl_accumulator[name]['In-Degree']),
        'Out-Degree': sum(overall_kl_accumulator[name]['Out-Degree']) / len(overall_kl_accumulator[name]['Out-Degree']),
        'WCC': sum(overall_kl_accumulator[name]['WCC']) / len(overall_kl_accumulator[name]['WCC']),
        'SCC': sum(overall_kl_accumulator[name]['SCC']) / len(overall_kl_accumulator[name]['SCC']),
        'HOPS': sum(overall_kl_accumulator[name]['HOPS']) / len(overall_kl_accumulator[name]['HOPS']),
        'sng-val': sum(overall_kl_accumulator[name]['sng-val']) / len(overall_kl_accumulator[name]['sng-val']),
        'sng-vec': sum(overall_kl_accumulator[name]['sng-vec']) / len(overall_kl_accumulator[name]['sng-vec']),
        'clustering': np.nanmean(overall_kl_accumulator[name]['clustering'])

    }

# Display the final overall average KS statistics
print("\nFinal Overall Average KS Statistics:")
for name, stats in final_average_results.items():
    print(f"{name}: In-Degree Average KS = {stats['In-Degree']}, Out-Degree Average KS = {stats['Out-Degree']}")
    print(f"{name}: WCC Average ks ={stats['WCC']}, SCC Average ks ={stats['SCC']}")
    print(f"{name}: HOP Average ks ={stats['HOPS']}, {name}: sng-val Average ks ={stats['sng-val']}")
    print(f"{name}: sng-vec Average ks ={stats['sng-vec']}, clustering Average ks ={stats['clustering']}")

print("\nFinal Overall Average KL Statistics:")
for name, stats in final_average_results_kl.items():
    print(f"{name}: In-Degree Average kl = {stats['In-Degree']}, Out-Degree Average Kl = {stats['Out-Degree']}")
    print(f"{name}: WCC Average kl ={stats['WCC']}, SCC Average kl ={stats['SCC']}")
    print(f"{name}: HOP Average kl ={stats['HOPS']}, {name}: sng-val Average kl ={stats['sng-val']}")
    print(f"{name}: sng-vec Average kl ={stats['sng-vec']}, clustering Average kl ={stats['clustering']}")

Running sampling for 60% of the original graph (Sample size: 11262)
------------------------------
Run results for Sampling method RN sample size 11262 and 60%:
In-Degree KS: 0.13867985526251442
Out-Degree KS: 0.13867985526251442
WCC KS: 0.15901839588533517
SCC KS: 0.15901839588533517
Hops KS: 0.3333333333333333
Sng-Value KS: 0.024226207084932172
Sng-Vec KS: 0.044444444444444446
Clustering KS: 0.04156954156954157
------------------------------
------------------------------
Run results for Sampling method RN sample size 11262 and 60%:
In-Degree Kl: 4.661020670882697
Out-Degree Kl: 4.661020670882697
WCC Kl: 0.0014437046000612239
SCC Kl: 0.0014437046000612239
Hops Kl: 8.965495012656088e-11
Sng-Value Kl: 0.006746537979967227
Sng-Vec Kl: 8.165498924667038
Clustering Kl: 0.2598646102394129
------------------------------
------------------------------
Run results for Sampling method RPN sample size 11262 and 60%:
In-Degree KS: 0.21978493534656507
Out-Degree KS: 0.21978493534656507
WCC KS: 0.